# Lab 8 · LLMs para analizar reclamaciones y tickets

Notebook para extraer información estructurada de tickets de texto libre. Incluye dos modos:

1. **Bedrock**: invoca un modelo fundacional si está disponible en AWS Academy.
2. **Alternativa local**: extractor por reglas para poder completar el aprendizaje si Bedrock no está habilitado.

El objetivo no es solo obtener etiquetas, sino validar la extracción y detectar alucinaciones o errores.

In [ ]:
%pip -q install pandas numpy boto3 s3fs

import os, json, time, re
from pathlib import Path
import pandas as pd
import numpy as np

USE_S3 = False
DATA_DIR = Path('data')
S3_PREFIX = 's3://TU_BUCKET/text'
USE_BEDROCK = False   # Cambia a True si Bedrock está habilitado en tu AWS Academy.
MODEL_ID = 'amazon.titan-text-express-v1'  # Sustituye por el modelId disponible.
REGION_NAME = 'us-east-1'

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

def data_path(filename):
    return f"{S3_PREFIX}/{filename}" if USE_S3 else str(DATA_DIR / filename)

## Parte A. Cargar tickets

In [ ]:
df = pd.read_csv(data_path('telco_tickets_text_lab8.csv'))
print(df.shape)
display(df.head(10))

## Parte B. Llamada a Bedrock o alternativa local

In [ ]:
if USE_BEDROCK:
    import boto3
    bedrock = boto3.client('bedrock-runtime', region_name=REGION_NAME)

    def preguntar(prompt, max_tokens=512):
        cuerpo = {
            "inputText": prompt,
            "textGenerationConfig": {"maxTokenCount": max_tokens, "temperature": 0.0}
        }
        resp = bedrock.invoke_model(modelId=MODEL_ID, body=json.dumps(cuerpo))
        out = json.loads(resp['body'].read())
        return out['results'][0]['outputText']
else:
    print('Modo local activado: no se invoca Bedrock. Cambia USE_BEDROCK=True para usar el modelo.')

## Parte C. Prompt de extracción estructurada

In [ ]:
PLANTILLA = """Eres un analista de atención al cliente de una operadora.
Extrae del siguiente ticket la información pedida y responde SOLO con un objeto JSON, sin texto adicional, con estas claves exactas:
- tema: uno de [averia, velocidad, facturacion, portabilidad, instalacion, consulta, baja, felicitacion, otro]
- servicio_afectado: uno de [internet, movil, fijo, television, varios, ninguno]
- urgencia: uno de [baja, media, alta, critica]
- sentimiento: uno de [positivo, neutral, negativo]
- posible_causa: texto breve SOLO si aparece explícitamente en el ticket; si no aparece, escribe "no especificada".
No inventes información que no esté en el texto.
TICKET:
\"\"\"{texto}\"\"\"
"""

print(PLANTILLA.format(texto=df.iloc[0]['text'])[:900])

## Parte D. Extractor local de respaldo

Sirve para ejecutar el lab si Bedrock no está disponible. Es deliberadamente más simple que un LLM, lo que permite comparar ventajas y limitaciones.

In [ ]:
def extractor_local(texto):
    t = texto.lower()
    if any(k in t for k in ['felicitar','gracias','muy buen servicio','amable']):
        tema='felicitacion'; sentimiento='positivo'
    elif any(k in t for k in ['baja','cancelar','dar de baja']):
        tema='baja'; sentimiento='negativo'
    elif any(k in t for k in ['factura','cobrado','cargo','devolución','precio']):
        tema='facturacion'; sentimiento='negativo'
    elif any(k in t for k in ['portabilidad','número móvil','traer dos líneas']):
        tema='portabilidad'; sentimiento='negativo' if any(k in t for k in ['sin servicio','nadie']) else 'neutral'
    elif any(k in t for k in ['instalación','instalar','técnico no acudió','cita']):
        tema='instalacion'; sentimiento='negativo' if any(k in t for k in ['no acudió','pendiente']) else 'neutral'
    elif any(k in t for k in ['lenta','velocidad','mb','teletrabajar']):
        tema='velocidad'; sentimiento='negativo'
    elif any(k in t for k in ['sin internet','no funciona','caído','avería','incidencia']):
        tema='averia'; sentimiento='negativo'
    elif any(k in t for k in ['quiero saber','información','permanencia','tarifa']):
        tema='consulta'; sentimiento='neutral'
    else:
        tema='otro'; sentimiento='neutral'

    if any(k in t for k in ['internet','fibra','router','conexión']): servicio='internet'
    elif any(k in t for k in ['móvil','línea móvil','número móvil','datos móviles']): servicio='movil'
    elif any(k in t for k in ['fija','fijo']): servicio='fijo'
    elif any(k in t for k in ['televisión','streaming','fútbol']): servicio='television'
    elif tema in ['facturacion','baja','consulta']: servicio='varios'
    else: servicio='ninguno'

    if any(k in t for k in ['crítico','sin servicio','urgente','trabajo','teletrabajar']): urgencia='alta'
    elif tema in ['averia','velocidad','facturacion','portabilidad','instalacion','baja']: urgencia='media'
    else: urgencia='baja'

    causa = 'incidencia en la zona' if 'incidencia en mi zona' in t else 'no especificada'
    return {'tema': tema, 'servicio_afectado': servicio, 'urgencia': urgencia, 'sentimiento': sentimiento, 'posible_causa': causa}

## Parte E. Función de extracción

In [ ]:
def extraer(texto):
    if not USE_BEDROCK:
        return extractor_local(texto)
    salida = preguntar(PLANTILLA.format(texto=texto))
    ini, fin = salida.find('{'), salida.rfind('}')
    try:
        return json.loads(salida[ini:fin+1])
    except Exception:
        return {'tema':'error', 'servicio_afectado':'error', 'urgencia':'error', 'sentimiento':'error', 'posible_causa':'error_parseo'}

print(extraer(df.iloc[0]['text']))

## Parte F. Procesar lote de tickets

In [ ]:
N = 30  # en Bedrock empieza con muestras pequeñas para controlar coste.
muestra = df.head(N).copy()
resultados = []
for _, fila in muestra.iterrows():
    resultados.append(extraer(fila['text']))
    if USE_BEDROCK:
        time.sleep(0.3)

ext = pd.DataFrame(resultados)
salida = pd.concat([muestra[['ticket_id','region','channel','text']].reset_index(drop=True), ext], axis=1)
display(salida.head(10))
salida.to_csv(OUTPUT_DIR/'lab08_extraccion_tickets.csv', index=False)

## Parte G. Análisis consolidado

In [ ]:
display(salida['tema'].value_counts().rename_axis('tema').reset_index(name='tickets'))
display(salida['urgencia'].value_counts().rename_axis('urgencia').reset_index(name='tickets'))
display(salida.groupby('region')['sentimiento'].value_counts().unstack(fill_value=0))

## Parte H. Validación contra revisión humana

El ZIP incluye `mi_anotacion_ejemplo.csv` con una anotación humana de ejemplo de los primeros 10 tickets. En clase puedes sustituirlo por la anotación hecha por el alumno.

In [ ]:
humano = pd.read_csv(data_path('mi_anotacion_ejemplo.csv'))
comparado = salida.merge(humano, on='ticket_id')
acierto_tema = (comparado['tema'] == comparado['tema_humano']).mean()
acierto_sent = (comparado['sentimiento'] == comparado['sentimiento_humano']).mean()
print(f'Acuerdo en tema: {acierto_tema:.0%}')
print(f'Acuerdo en sentimiento: {acierto_sent:.0%}')
display(comparado[['ticket_id','text','tema','tema_humano','sentimiento','sentimiento_humano','posible_causa']])
comparado.to_csv(OUTPUT_DIR/'lab08_validacion_humana.csv', index=False)

## Cierre

Para el entregable, incluye el prompt, la tabla de extracción, los resultados agregados, la validación manual y tres errores o riesgos: alucinación, ironía y categoría forzada.